# Public Library Survey × ACS Socioeconomic Analysis
**Data sources:**
- *Public Library Survey (PLS) FY2023* — IMLS agency-level and outlet-level microdata
- *American Community Survey (ACS) 5-Year Estimates 2023, DP03* — county-level economic characteristics

**Project timeline covered in this notebook:**
| Phase | Dates | Section |
|---|---|---|
| Data Cleaning & Preprocessing | Mar 24 – 27 | §1 |
| Exploratory Data Analysis | Mar 27 – 30 | §2 |
| Feature Engineering | Apr 1 – 6 | §3 |
| Data Quality & Documentation | Apr 7 – 12 | §4 |


## §0 — Imports & File Paths

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Display settings ──────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# ── File paths ────────────────────────────────────────────────
# Update these paths if you move the data files.
PATH_ACS_META  = "ACSDP5Y2023_DP03-Column-Metadata.csv"
PATH_ACS_DATA  = "ACSDP5Y2023_DP03-Data.csv"
PATH_PLS_AE    = "PLS_FY23_AE_pud23i.csv"      # Agency-level (one row per library system)
PATH_PLS_OUT   = "pls_fy23_outlet_pud23i.csv"   # Outlet-level (branches / bookmobiles)

print("Imports OK")


---
## §1 — Data Cleaning & Preprocessing
*Mar 24 – 27*

### 1.1 Load raw files


In [ ]:
# ── ACS metadata (column code → human label) ─────────────────
acs_meta = pd.read_csv(PATH_ACS_META)
print(f"ACS metadata: {acs_meta.shape[0]} column definitions")
acs_meta.head()


In [ ]:
# ── ACS data ─────────────────────────────────────────────────
# Row 0 is a second header row (human-readable labels); skip it with skiprows=[1].
acs_raw = pd.read_csv(PATH_ACS_DATA, skiprows=[1], low_memory=False)
print(f"ACS: {acs_raw.shape[0]:,} counties × {acs_raw.shape[1]} columns")
acs_raw.head(2)


In [ ]:
# ── PLS agency-level ──────────────────────────────────────────
pls_raw = pd.read_csv(PATH_PLS_AE, low_memory=False)
print(f"PLS agencies: {pls_raw.shape[0]:,} rows × {pls_raw.shape[1]} columns")
pls_raw.head(2)


In [ ]:
# ── PLS outlet-level ──────────────────────────────────────────
outlet_raw = pd.read_csv(PATH_PLS_OUT, low_memory=False)
print(f"PLS outlets: {outlet_raw.shape[0]:,} rows × {outlet_raw.shape[1]} columns")
outlet_raw.head(2)


### 1.2 Select relevant ACS columns (socioeconomic predictors)

In [ ]:
# ── Build a lookup dict from the metadata ────────────────────
meta_lookup = dict(zip(acs_meta["Column Name"], acs_meta["Label"]))

# ── Priority socioeconomic predictor columns ──────────────────
# We keep Estimate (E) columns for key indicators and their Percent (PE) variants.
# Margin-of-error (M/PM) columns are dropped for now; add them back if doing
# uncertainty-aware modeling.

ACS_KEEP = {
    # Employment
    "DP03_0005PE": "unemployment_rate_pct",
    # Income
    "DP03_0062E":  "median_hh_income",
    "DP03_0063E":  "mean_hh_income",
    "DP03_0086E":  "per_capita_income",
    # Poverty
    "DP03_0099PE": "poverty_rate_all_pct",
    "DP03_0100PE": "poverty_rate_under18_pct",
    "DP03_0101PE": "poverty_rate_18to64_pct",
    # SNAP / food assistance
    "DP03_0074PE": "snap_pct",
    # Health insurance
    "DP03_0096PE": "no_health_insurance_pct",
    # Work-from-home
    "DP03_0042PE": "work_from_home_pct",
}

# Always keep geography columns
GEO_COLS = ["GEO_ID", "NAME"]

acs_sel = acs_raw[GEO_COLS + list(ACS_KEEP.keys())].copy()
acs_sel = acs_sel.rename(columns=ACS_KEEP)
print(f"ACS subset: {acs_sel.shape}")
acs_sel.head(3)


In [ ]:
# ── Decode ACS GEO_ID to 5-digit county FIPS ─────────────────
# GEO_ID format: "0500000US{STATE_FIPS}{COUNTY_FIPS}"  e.g. "0500000US01001"
acs_sel["county_fips"] = acs_sel["GEO_ID"].str.extract(r"US(\d{5})$")

# Drop the full GEO_ID (redundant once we have county_fips)
acs_sel = acs_sel.drop(columns=["GEO_ID"])
print(f"Parsed FIPS codes: {acs_sel['county_fips'].notna().sum():,} non-null")
acs_sel[["NAME", "county_fips"]].head(5)


### 1.3 Select relevant PLS columns (library usage metrics)

The PLS agency file has 187 columns. We keep identity/geography fields, core usage
metrics, and financial variables. Flag-columns (e.g. `F_VISITS`) encode data quality
codes — we'll use them to identify suppressed or imputed values.


In [ ]:
# ── Key PLS agency columns ───────────────────────────────────
PLS_KEEP = {
    # Identity
    "STABR":     "state",
    "FSCSKEY":   "fscskey",
    "LIBID":     "libid",
    "LIBNAME":   "lib_name",
    "CNTY":      "county_name",
    # Geography / linkage
    "LSAGEOID":  "lsa_geoid",
    "LSAGEOTYPE":"lsa_geotype",
    "LONGITUD":  "longitude",
    "LATITUDE":  "latitude",
    # Population served
    "POPU_LSA":  "pop_lsa",
    # Staffing
    "LIBRARIA":  "librarians_fte",
    "OTHPAID":   "other_paid_fte",
    "TOTSTAFF":  "total_staff_fte",
    # Financials
    "LOCGVT":    "local_govt_revenue",
    "STGVT":     "state_govt_revenue",
    "FEDGVT":    "federal_revenue",
    "TOTINCM":   "total_income",
    "SALARIES":  "salaries_exp",
    "TOTOPEXP":  "total_opex",
    # Collections
    "BKVOL":     "print_volumes",
    "EBOOK":     "ebook_titles",
    # Usage — visits & circulation
    "VISITS":    "visits",
    "TOTCIR":    "total_circ",
    "KIDCIRCL":  "kids_circ",
    "ELMATCIR":  "elec_mat_circ",
    "REGBOR":    "registered_borrowers",
    # Digital / reference
    "ELINFO":    "elec_info_retrievals",
    "ELCONT":    "elec_content_uses",
    "GPTERMS":   "public_internet_terminals",
    "PITUSR":    "internet_uses",
    "WIFISESS":  "wifi_sessions",
    # Programs & attendance
    "TOTPRO":    "total_programs",
    "TOTATTEN":  "total_attendance",
    "ONPRO":     "online_programs",
    "OFFPRO":    "inperson_programs",
    # Hours
    "HRS_OPEN":  "hrs_open",
    # Outlets
    "CENTLIB":   "central_libs",
    "BRANLIB":   "branch_libs",
    "BKMOB":     "bookmobiles",
    # Quality flags (keep for filtering)
    "F_VISITS":  "f_visits",
    "F_TOTCIR":  "f_totcir",
    "F_REGBOR":  "f_regbor",
    "F_TOTSTF":  "f_totstf",
}

pls = pls_raw[list(PLS_KEEP.keys())].copy().rename(columns=PLS_KEEP)
print(f"PLS subset: {pls.shape}")
pls.head(2)


### 1.4 Handle PLS sentinel values and missing data

In [ ]:
# ── PLS encodes several special values ───────────────────────
# -1  = not applicable
# -3  = confidential / suppressed
# -9  = missing / not reported
# These appear in numeric columns and should become NaN.

PLS_SENTINELS = [-1, -3, -9]

NUMERIC_COLS = [
    "pop_lsa", "librarians_fte", "other_paid_fte", "total_staff_fte",
    "local_govt_revenue", "state_govt_revenue", "federal_revenue",
    "total_income", "salaries_exp", "total_opex",
    "print_volumes", "ebook_titles",
    "visits", "total_circ", "kids_circ", "elec_mat_circ",
    "registered_borrowers", "elec_info_retrievals", "elec_content_uses",
    "public_internet_terminals", "internet_uses", "wifi_sessions",
    "total_programs", "total_attendance", "online_programs", "inperson_programs",
    "hrs_open", "central_libs", "branch_libs", "bookmobiles",
]

before = pls[NUMERIC_COLS].isin(PLS_SENTINELS).sum().sum()
pls[NUMERIC_COLS] = pls[NUMERIC_COLS].replace(PLS_SENTINELS, np.nan)
after = pls[NUMERIC_COLS].isna().sum().sum()
print(f"Sentinel values replaced → {before:,} cells now NaN")


In [ ]:
# ── ACS: replace ACS sentinel values ─────────────────────────
# ACS uses "(X)" for not applicable, "-" for zero-base percent, etc.
ACS_NUMERIC = list(ACS_KEEP.values())

for col in ACS_NUMERIC:
    acs_sel[col] = pd.to_numeric(acs_sel[col], errors="coerce")

# Quick missingness report
missing_acs = acs_sel[ACS_NUMERIC].isna().mean().mul(100).round(2)
print("ACS missing % per column:")
print(missing_acs.to_string())


In [ ]:
# ── PLS missingness overview ─────────────────────────────────
missing_pls = pls[NUMERIC_COLS].isna().mean().mul(100).round(1).sort_values(ascending=False)
print("PLS missing % per column (top 20):")
print(missing_pls.head(20).to_string())


### 1.5 Build the county-level FIPS key for PLS and merge

In [ ]:
# ── Derive county FIPS from PLS ──────────────────────────────
# Strategy:
#   • When LSAGEOTYPE == "COUNTY"  → pad lsa_geoid to 5 digits = county FIPS directly.
#   • When LSAGEOTYPE == "PLACE"   → the library serves a municipality; we can't map
#     directly to one county without a crosswalk. We fall back to CENTRACT, where the
#     first 5 digits of the 11-digit census tract ID are the county FIPS.
#
# NOTE: Some libraries span multiple counties (LSABOUND == "Y"). This merge assigns
# them to a single county — document this limitation in §4.

def derive_county_fips(row):
    if row["lsa_geotype"] == "COUNTY":
        # geoid stored as integer; zero-pad to 5 digits
        try:
            return str(int(row["lsa_geoid"])).zfill(5)
        except (ValueError, TypeError):
            return np.nan
    return np.nan  # handled next step

pls["county_fips_direct"] = pls.apply(derive_county_fips, axis=1)

print("Direct county FIPS (COUNTY type):",
      pls["county_fips_direct"].notna().sum(), "/", len(pls))
print("Remaining (PLACE/other):", pls["county_fips_direct"].isna().sum())


In [ ]:
# ── For PLACE-type libraries: extract county from LSAGEOID ───
# Census place GEOIDs are 7 digits: state(2) + place(5).
# We use LSAGEOID to look up county via CENTRACT if available in the agency file.
#
# Simpler fallback approach used here: re-read the raw data to get CENTRACT,
# then take the first 5 digits.

pls_raw_geo = pls_raw[["FSCSKEY", "CENTRACT", "LSAGEOTYPE"]].copy()
pls_raw_geo["CENTRACT"] = pls_raw_geo["CENTRACT"].astype(str).str.zfill(11)
pls_raw_geo["county_fips_tract"] = pls_raw_geo["CENTRACT"].str[:5]
# Only use this for non-county types (validity check: should be 5 digit numeric)
pls_raw_geo.loc[~pls_raw_geo["county_fips_tract"].str.match(r"^\d{5}$"), "county_fips_tract"] = np.nan

# Merge back
pls = pls.merge(pls_raw_geo[["FSCSKEY", "county_fips_tract"]], on="FSCSKEY", how="left")

# Combine: prefer direct COUNTY match, fall back to tract-derived
pls["county_fips"] = pls["county_fips_direct"].fillna(pls["county_fips_tract"])

n_matched = pls["county_fips"].notna().sum()
print(f"Library agencies with county FIPS: {n_matched:,} / {len(pls):,} ({n_matched/len(pls)*100:.1f}%)")


In [ ]:
# ── Merge PLS × ACS on county FIPS ───────────────────────────
merged = pls.merge(acs_sel, on="county_fips", how="left", suffixes=("_pls", "_acs"))

print(f"Merged dataset: {merged.shape}")
print(f"Libraries with ACS match: {merged['median_hh_income'].notna().sum():,} / {len(merged):,}")


In [ ]:
# ── Inspect merge quality ─────────────────────────────────────
# Libraries without a county ACS match (investigate)
unmatched = merged[merged["median_hh_income"].isna()][["lib_name", "state", "county_name", "lsa_geotype"]]
print("Unmatched sample:")
print(unmatched.head(10).to_string(index=False))
print("\nGeotype breakdown of unmatched:")
print(merged[merged["median_hh_income"].isna()]["lsa_geotype"].value_counts())


In [ ]:
# ── Save cleaned files ────────────────────────────────────────
pls.to_csv("pls_cleaned.csv", index=False)
acs_sel.to_csv("acs_cleaned.csv", index=False)
merged.to_csv("merged_pls_acs.csv", index=False)
print("Saved: pls_cleaned.csv, acs_cleaned.csv, merged_pls_acs.csv")


---
## §2 — Exploratory Data Analysis
*Mar 27 – 30*

### 2.1 Summary statistics


In [ ]:
# Work on the merged dataset, restricting to rows that have valid population
df = merged[merged["pop_lsa"].notna() & (merged["pop_lsa"] > 0)].copy()
print(f"Working dataset: {len(df):,} libraries")


In [ ]:
# ── Summary stats for library usage metrics ──────────────────
USAGE_COLS = ["visits", "total_circ", "registered_borrowers",
              "total_programs", "total_attendance",
              "hrs_open", "public_internet_terminals", "wifi_sessions",
              "total_staff_fte", "total_opex", "pop_lsa"]

df[USAGE_COLS].describe().T.round(1)


In [ ]:
# ── Summary stats for ACS socioeconomic predictors ───────────
SOC_COLS = list(ACS_KEEP.values())

acs_sel[SOC_COLS].describe().T.round(2)


### 2.2 Distribution checks — skewness & outliers

In [ ]:
# ── Skewness table for usage metrics ─────────────────────────
skew_df = df[USAGE_COLS].apply(lambda c: pd.Series({
    "mean":     c.mean(),
    "median":   c.median(),
    "skewness": c.skew(),
    "kurtosis": c.kurt(),
    "pct_zero": (c == 0).mean() * 100,
    "n_valid":  c.notna().sum(),
})).T.round(3)

print("Skewness summary (|skew| > 1 = substantially skewed):")
skew_df.sort_values("skewness", ascending=False)


In [ ]:
# ── Distribution plots for key usage variables (raw vs log) ──
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

plot_vars = ["visits", "total_circ", "registered_borrowers",
             "total_programs", "total_attendance", "wifi_sessions",
             "public_internet_terminals", "total_opex"]

for i, col in enumerate(plot_vars):
    ax_raw = axes[i * 2 // 4 * 2 + i % 4]   # simplified: just use flat index
    pass  # replaced below with cleaner loop

fig.clear()
fig, axes = plt.subplots(len(plot_vars), 2, figsize=(14, len(plot_vars) * 3))

for i, col in enumerate(plot_vars):
    series = df[col].dropna()
    # Raw
    axes[i, 0].hist(series, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
    axes[i, 0].set_title(f"{col} (raw)", fontsize=10)
    axes[i, 0].set_xlabel(col); axes[i, 0].set_ylabel("Count")
    # Log-transformed
    log_series = np.log1p(series)
    axes[i, 1].hist(log_series, bins=60, color="coral", edgecolor="none", alpha=0.8)
    axes[i, 1].set_title(f"log1p({col})", fontsize=10)
    axes[i, 1].set_xlabel("log1p value"); axes[i, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("fig_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: fig_distributions.png")


In [ ]:
# ── Outlier detection via IQR ─────────────────────────────────
def iqr_outlier_pct(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)
    return mask.mean() * 100

outlier_pct = df[USAGE_COLS].apply(lambda c: iqr_outlier_pct(c.dropna()))
outlier_pct = outlier_pct.rename("outlier_pct_iqr").sort_values(ascending=False)
print("Outlier % by column (IQR ×1.5 rule):")
print(outlier_pct.to_string())


### 2.3 Geographic overview — state-level medians

In [ ]:
# ── State-level aggregation ───────────────────────────────────
state_summary = df.groupby("state").agg(
    n_libraries         = ("fscskey", "count"),
    median_visits       = ("visits", "median"),
    median_circ         = ("total_circ", "median"),
    median_income_served= ("median_hh_income", "median"),
    median_poverty_pct  = ("poverty_rate_all_pct", "median"),
).reset_index().sort_values("median_visits", ascending=False)

print("Top 10 states by median library visits:")
state_summary.head(10).to_string(index=False)


### 2.4 Correlation matrix — socioeconomic predictors × library usage

In [ ]:
# ── Select columns with enough valid data ────────────────────
CORR_USAGE = ["visits", "total_circ", "registered_borrowers",
              "total_programs", "total_attendance",
              "hrs_open", "wifi_sessions", "internet_uses"]

CORR_SOC   = ["median_hh_income", "poverty_rate_all_pct",
              "unemployment_rate_pct", "per_capita_income",
              "snap_pct", "no_health_insurance_pct",
              "work_from_home_pct"]

corr_df = df[CORR_USAGE + CORR_SOC].copy()

# Log-transform usage (right-skewed)
for col in CORR_USAGE:
    corr_df[f"log_{col}"] = np.log1p(corr_df[col])

log_usage = [f"log_{c}" for c in CORR_USAGE]
corr_matrix = corr_df[log_usage + CORR_SOC].corr(method="spearman")

# ── Plot: only the usage × socioeconomic block ───────────────
cross_corr = corr_matrix.loc[log_usage, CORR_SOC]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(cross_corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax,
            yticklabels=[c.replace("log_", "") for c in log_usage])
ax.set_title("Spearman Correlation: Library Usage (log) × Socioeconomic Predictors", fontsize=13)
plt.tight_layout()
plt.savefig("fig_correlation_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_correlation_heatmap.png")


In [ ]:
# ── Highlight strongest correlations ─────────────────────────
cross_long = cross_corr.stack().reset_index()
cross_long.columns = ["usage_metric", "socio_predictor", "spearman_r"]
cross_long["abs_r"] = cross_long["spearman_r"].abs()
print("Top 15 correlations (by |r|):")
print(cross_long.sort_values("abs_r", ascending=False).head(15).to_string(index=False))


In [ ]:
# ── Scatter: median household income vs. visits/capita (preview) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (x, y, title) in zip(axes, [
    ("median_hh_income", "visits",     "Median HH Income vs. Visits"),
    ("poverty_rate_all_pct", "total_circ", "Poverty Rate vs. Total Circulation"),
]):
    sub = df[[x, y]].dropna()
    ax.scatter(sub[x], np.log1p(sub[y]), alpha=0.25, s=10, color="steelblue")
    # Trend line
    m, b, r, p, _ = stats.linregress(sub[x], np.log1p(sub[y]))
    xr = np.linspace(sub[x].min(), sub[x].max(), 200)
    ax.plot(xr, m * xr + b, color="firebrick", lw=2, label=f"r={r:.2f}, p={p:.3f}")
    ax.set_xlabel(x); ax.set_ylabel(f"log1p({y})")
    ax.set_title(title); ax.legend()

plt.tight_layout()
plt.savefig("fig_scatter_preview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: fig_scatter_preview.png")


---
## §3 — Feature Engineering
*Apr 1 – 6*

### 3.1 Per-capita conversion
Dividing raw counts by population of legal service area (POPU_LSA) makes
libraries of different sizes comparable. A small rural library serving 1,000
people looks very different in raw counts vs. a large urban system serving 500,000.


In [ ]:
fe = merged[merged["pop_lsa"].notna() & (merged["pop_lsa"] > 0)].copy()

# ── Per-capita usage metrics ──────────────────────────────────
PC_PAIRS = [
    ("visits",               "visits_pc"),
    ("total_circ",           "circ_pc"),
    ("registered_borrowers", "regbor_pc"),
    ("total_programs",       "programs_pc"),
    ("total_attendance",     "attendance_pc"),
    ("hrs_open",             "hrs_open_pc"),
    ("public_internet_terminals", "terminals_pc"),
    ("wifi_sessions",        "wifi_pc"),
    ("total_opex",           "opex_pc"),
    ("total_income",         "income_pc"),
    ("total_staff_fte",      "staff_fte_pc"),
]

for raw_col, pc_col in PC_PAIRS:
    fe[pc_col] = fe[raw_col] / fe["pop_lsa"]

PC_COLS = [pc for _, pc in PC_PAIRS]
print("Per-capita columns created:")
fe[PC_COLS].describe().T.round(4)


### 3.2 Log-transform skewed variables

In [ ]:
# ── log1p transform on per-capita and raw count columns ──────
LOG_TARGETS = PC_COLS + ["visits", "total_circ", "total_opex",
                          "median_hh_income", "per_capita_income"]

for col in LOG_TARGETS:
    if col in fe.columns:
        fe[f"log_{col}"] = np.log1p(fe[col].clip(lower=0))

print("Log-transformed columns added:", sum(1 for c in fe.columns if c.startswith("log_")))


### 3.3 Derived ratio / composite features

In [ ]:
# ── Financial efficiency ratios ──────────────────────────────
fe["cost_per_visit"]    = fe["total_opex"] / fe["visits"].replace(0, np.nan)
fe["cost_per_circ"]     = fe["total_opex"] / fe["total_circ"].replace(0, np.nan)
fe["circ_per_visit"]    = fe["total_circ"] / fe["visits"].replace(0, np.nan)
fe["digital_share_circ"]= fe["elec_mat_circ"] / fe["total_circ"].replace(0, np.nan)

# ── Staffing ratio ────────────────────────────────────────────
fe["staff_per_1k_pop"]  = fe["total_staff_fte"] / fe["pop_lsa"] * 1000

# ── Program intensity ─────────────────────────────────────────
fe["attendance_per_program"] = fe["total_attendance"] / fe["total_programs"].replace(0, np.nan)

# ── Digital engagement index (simple sum of z-scores) ─────────
digital_vars = ["internet_uses", "wifi_sessions", "elec_mat_circ", "elec_content_uses"]
for v in digital_vars:
    if v in fe.columns:
        fe[f"z_{v}"] = (fe[v] - fe[v].mean()) / fe[v].std()

z_cols = [f"z_{v}" for v in digital_vars if v in fe.columns]
if z_cols:
    fe["digital_engagement_idx"] = fe[z_cols].mean(axis=1)

print("Derived feature columns:")
derived = ["cost_per_visit", "cost_per_circ", "circ_per_visit",
           "digital_share_circ", "staff_per_1k_pop",
           "attendance_per_program", "digital_engagement_idx"]
fe[derived].describe().T.round(3)


### 3.4 Winsorize to reduce outlier influence

In [ ]:
from scipy.stats import mstats

def winsorize_col(series, limits=(0.01, 0.01)):
    """Winsorize at 1st and 99th percentile."""
    clean = series.dropna()
    if len(clean) == 0:
        return series
    arr = mstats.winsorize(clean, limits=limits)
    result = series.copy()
    result[series.notna()] = arr
    return result

WIN_COLS = PC_COLS + derived
for col in WIN_COLS:
    if col in fe.columns:
        fe[f"w_{col}"] = winsorize_col(fe[col])

print(f"Winsorized columns created: {sum(1 for c in fe.columns if c.startswith('w_'))}")


In [ ]:
# ── Distribution comparison: raw vs winsorized visits/capita ─
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fe["visits_pc"].dropna().hist(bins=80, ax=axes[0], color="steelblue", alpha=0.8)
axes[0].set_title("visits_pc (raw)"); axes[0].set_xlabel("Visits per capita")
fe["w_visits_pc"].dropna().hist(bins=80, ax=axes[1], color="coral", alpha=0.8)
axes[1].set_title("visits_pc (winsorized 1–99%)"); axes[1].set_xlabel("Visits per capita")
plt.tight_layout()
plt.savefig("fig_winsorize_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Save feature-engineered dataset ──────────────────────────
fe.to_csv("features_pls_acs.csv", index=False)
print(f"Saved features_pls_acs.csv: {fe.shape[0]:,} rows × {fe.shape[1]} columns")


---
## §4 — Data Quality & Reproducibility Documentation
*Apr 7 – 12*

This section produces structured documentation of the data pipeline for inclusion
in the final project report.

### 4.1 Data provenance metadata


In [ ]:
import datetime, hashlib, os

provenance = {
    "generated_at": datetime.datetime.now().isoformat(),
    "files": {
        "PLS_FY23_AE_pud23i.csv": {
            "source": "IMLS Public Library Survey FY2023 Agency-level microdata",
            "url": "https://www.imls.gov/research-evaluation/data-collection/public-libraries-survey",
            "year": 2023,
            "unit_of_obs": "library_agency",
            "n_rows": len(pls_raw),
            "n_cols": pls_raw.shape[1],
        },
        "pls_fy23_outlet_pud23i.csv": {
            "source": "IMLS Public Library Survey FY2023 Outlet-level microdata",
            "url": "https://www.imls.gov/research-evaluation/data-collection/public-libraries-survey",
            "year": 2023,
            "unit_of_obs": "library_outlet",
            "n_rows": len(outlet_raw),
            "n_cols": outlet_raw.shape[1],
        },
        "ACSDP5Y2023_DP03-Data.csv": {
            "source": "U.S. Census Bureau ACS 5-Year Estimates 2023, Table DP03",
            "url": "https://data.census.gov",
            "survey_years": "2019–2023",
            "unit_of_obs": "county (FIPS)",
            "n_rows": len(acs_raw),
            "n_cols": acs_raw.shape[1],
        },
    }
}

import json
print(json.dumps(provenance, indent=2))


### 4.2 Column-level data dictionary

In [ ]:
# ── Build a data dictionary for the merged/feature dataset ───
def col_profile(series, label=""):
    return {
        "label":      label,
        "dtype":      str(series.dtype),
        "n_valid":    int(series.notna().sum()),
        "pct_missing":round(series.isna().mean() * 100, 1),
        "min":        round(float(series.min()), 4) if series.notna().any() else None,
        "max":        round(float(series.max()), 4) if series.notna().any() else None,
        "mean":       round(float(series.mean()), 4) if pd.api.types.is_numeric_dtype(series) else None,
        "median":     round(float(series.median()), 4) if pd.api.types.is_numeric_dtype(series) else None,
    }

DICT_COLS = {
    # PLS usage
    "visits":                "Total annual visits to all outlets",
    "total_circ":            "Total annual circulation (physical + digital)",
    "kids_circ":             "Children's circulation",
    "elec_mat_circ":         "Electronic materials circulation",
    "registered_borrowers":  "Registered borrowers (active cardholders)",
    "total_programs":        "Total programs offered (all age groups)",
    "total_attendance":      "Total attendance at library programs",
    "hrs_open":              "Total annual public service hours",
    "public_internet_terminals": "Number of public internet access terminals",
    "wifi_sessions":         "Public Wi-Fi sessions",
    "total_staff_fte":       "Total paid staff (FTE)",
    "total_opex":            "Total operating expenditures ($)",
    "total_income":          "Total revenue from all sources ($)",
    "pop_lsa":               "Population of legal service area",
    # ACS socioeconomic
    "median_hh_income":      "Median household income, ACS 5yr 2023 ($)",
    "poverty_rate_all_pct":  "% of all people below poverty line",
    "unemployment_rate_pct": "Civilian labor force unemployment rate (%)",
    "per_capita_income":     "Per capita income ($)",
    "snap_pct":              "% of households receiving SNAP benefits",
    "no_health_insurance_pct":"% of population without health insurance",
    "work_from_home_pct":    "% of workers who work from home",
    # Engineered
    "visits_pc":             "Annual visits per capita (pop_lsa denominator)",
    "circ_pc":               "Annual circulation per capita",
    "opex_pc":               "Operating expenditures per capita ($)",
    "cost_per_visit":        "Operating expenditure per visit ($)",
    "digital_share_circ":    "Proportion of circulation that is electronic",
    "staff_per_1k_pop":      "FTE staff per 1,000 population served",
    "digital_engagement_idx":"Composite z-score index of digital service metrics",
}

dict_rows = []
for col, label in DICT_COLS.items():
    if col in fe.columns:
        profile = col_profile(fe[col], label)
        profile["column"] = col
        dict_rows.append(profile)

data_dict = pd.DataFrame(dict_rows).set_index("column")
data_dict


In [ ]:
# ── Export data dictionary ────────────────────────────────────
data_dict.to_csv("data_dictionary.csv")
print("Saved: data_dictionary.csv")


### 4.3 Known limitations & methodological notes

In [ ]:
limitations = """
DATA QUALITY & METHODOLOGICAL NOTES
====================================

1. FIPS MERGE COVERAGE
   ~{pct_matched:.0f}% of PLS library agencies were matched to an ACS county record.
   Libraries that serve multiple counties (LSABOUND='Y') are assigned to a single
   county using their census tract centroid — a deliberate simplification.
   Libraries in unincorporated places or with missing CENTRACT data may be unmatched.

2. PLS SENTINEL VALUES
   The PLS encodes special non-response codes: -1 (not applicable), -3 (confidential),
   -9 (missing). All three are replaced with NaN before analysis. Flag columns
   (F_VISITS, F_TOTCIR, etc.) encode additional quality codes ('R' = revised,
   'H' = imputed, 'CT' = closed-then-reopened). These flags are retained in
   pls_cleaned.csv for reference but not yet used to filter the analysis.

3. ACS SURVEY UNCERTAINTY
   ACS 5-year estimates have margins of error, especially for small counties.
   Margin-of-error columns (suffix 'M') were dropped in §1 to keep the dataset
   manageable. For robust inference on small counties, restore and use them.

4. UNIT OF ANALYSIS
   The merged dataset is at the *library agency* level, not at the outlet (branch)
   level. Agencies vary enormously in size. Per-capita metrics (§3) partially address
   this but do not fully control for urban/rural differences in service area definition.

5. TIME ALIGNMENT
   PLS FY2023 covers fiscal year 2022–23 (varies by library).
   ACS 5-year 2023 estimates are averaged over 2019–2023.
   This temporal mismatch is standard practice in cross-sectional library research
   but means ACS figures reflect pre-pandemic norms mixed with pandemic-era data.

6. WINSORIZATION THRESHOLD
   Per-capita metrics are winsorized at the 1st and 99th percentiles. This is a
   conservative threshold; consider 5th/95th for smaller analytical subsets.
""".format(pct_matched=fe["median_hh_income"].notna().mean() * 100)

print(limitations)
with open("limitations_notes.txt", "w") as f:
    f.write(limitations)
print("Saved: limitations_notes.txt")


### 4.4 Reproducibility checklist

In [ ]:
import sys

print("=== REPRODUCIBILITY CHECKLIST ===")
print(f"Python version  : {sys.version.split()[0]}")
print(f"pandas          : {pd.__version__}")
print(f"numpy           : {np.__version__}")
print(f"matplotlib      : {plt.matplotlib.__version__}")
print(f"seaborn         : {sns.__version__}")
print()
print("=== OUTPUT FILES PRODUCED ===")
outputs = [
    "pls_cleaned.csv",
    "acs_cleaned.csv",
    "merged_pls_acs.csv",
    "features_pls_acs.csv",
    "data_dictionary.csv",
    "limitations_notes.txt",
    "fig_distributions.png",
    "fig_correlation_heatmap.png",
    "fig_scatter_preview.png",
    "fig_winsorize_comparison.png",
]
for f in outputs:
    exists = os.path.exists(f)
    print(f"  {'✓' if exists else '✗'}  {f}")

print()
print("To reproduce from scratch:")
print("  1. Place all four source CSVs in the same directory as this notebook.")
print("  2. Run all cells top-to-bottom (Kernel → Restart & Run All).")
print("  3. All outputs will be written to the working directory.")


### 4.5 File organization map

In [ ]:
org_map = """
project/
├── data/
│   ├── raw/                          ← original source files (do not edit)
│   │   ├── PLS_FY23_AE_pud23i.csv
│   │   ├── pls_fy23_outlet_pud23i.csv
│   │   ├── ACSDP5Y2023_DP03-Data.csv
│   │   └── ACSDP5Y2023_DP03-Column-Metadata.csv
│   └── processed/                    ← outputs of this notebook
│       ├── pls_cleaned.csv
│       ├── acs_cleaned.csv
│       ├── merged_pls_acs.csv        ← §1 merge output
│       └── features_pls_acs.csv     ← §3 feature-engineered dataset
├── docs/
│   ├── data_dictionary.csv           ← §4 column-level documentation
│   └── limitations_notes.txt         ← §4 methodological caveats
├── figures/
│   ├── fig_distributions.png
│   ├── fig_correlation_heatmap.png
│   ├── fig_scatter_preview.png
│   └── fig_winsorize_comparison.png
└── notebooks/
    └── 01_cleaning_eda_features.ipynb  ← this notebook
"""
print(org_map)
